In [1]:
import pandas as pd

In [2]:
# 📂 File paths for raw healthcare data (relative paths from repo root)
train_beneficiary_path = '../data/raw/Train_Beneficiarydata-1542865627584.csv'
train_inpatient_path = '../data/raw/Train_Inpatientdata-1542865627584.csv'
train_outpatient_path = '../data/raw/Train_Outpatientdata-1542865627584.csv'
train_path = '../data/raw/Train-1542865627584.csv'

test_beneficiary_path = '../data/raw/Test_Beneficiarydata-1542969243754.csv'
test_inpatient_path = '../data/raw/Test_Inpatientdata-1542969243754.csv'
test_outpatient_path = '../data/raw/Test_Outpatientdata-1542969243754.csv'
test_path = '../data/raw/Test-1542969243754.csv'

In [3]:
# Load training datasets
train_beneficiary = pd.read_csv(train_beneficiary_path)
train_inpatient = pd.read_csv(train_inpatient_path)
train_outpatient = pd.read_csv(train_outpatient_path)
train = pd.read_csv(train_path)

In [4]:
# Load test datasets
test_beneficiary = pd.read_csv(test_beneficiary_path)
test_inpatient = pd.read_csv(test_inpatient_path)
test_outpatient = pd.read_csv(test_outpatient_path)
test = pd.read_csv(test_path)

In [5]:
# ✅ View summary
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (5410, 2)
Test shape: (1353, 1)


In [6]:
print("Initial structure of train_beneficiary:")
train_beneficiary.info()

Initial structure of train_beneficiary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138556 entries, 0 to 138555
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   BeneID                           138556 non-null  object
 1   DOB                              138556 non-null  object
 2   DOD                              1421 non-null    object
 3   Gender                           138556 non-null  int64 
 4   Race                             138556 non-null  int64 
 5   RenalDiseaseIndicator            138556 non-null  object
 6   State                            138556 non-null  int64 
 7   County                           138556 non-null  int64 
 8   NoOfMonths_PartACov              138556 non-null  int64 
 9   NoOfMonths_PartBCov              138556 non-null  int64 
 10  ChronicCond_Alzheimer            138556 non-null  int64 
 11  ChronicCond_Heartfailure         13855

In [7]:
# CLEANING + FEATURE ENGINEERING
def data_cleaning_beneficiary(df):
    df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
    df['DOD'] = pd.to_datetime(df['DOD'], errors='coerce')
    
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
    df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

    categorical_cols = df.select_dtypes(include=['object']).columns
    df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])

    category_columns = ['Gender', 'Race', 'RenalDiseaseIndicator', 'State', 'County']
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')

    return df

In [8]:
def add_engineered_features(df):
    df['AgeAtDeathOrLastClaim'] = ((df['DOD'].fillna(pd.Timestamp('2020-01-01')) - df['DOB']).dt.days // 365)
    return df

# APPLY LOGIC
train_beneficiary = add_engineered_features(data_cleaning_beneficiary(train_beneficiary))
test_beneficiary = add_engineered_features(data_cleaning_beneficiary(test_beneficiary))

In [9]:
print("Post-cleaning structure of train_beneficiary:")
train_beneficiary.info()

Post-cleaning structure of train_beneficiary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138556 entries, 0 to 138555
Data columns (total 26 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   BeneID                           138556 non-null  object        
 1   DOB                              138556 non-null  datetime64[ns]
 2   DOD                              1421 non-null    datetime64[ns]
 3   Gender                           138556 non-null  category      
 4   Race                             138556 non-null  category      
 5   RenalDiseaseIndicator            138556 non-null  category      
 6   State                            138556 non-null  category      
 7   County                           138556 non-null  category      
 8   NoOfMonths_PartACov              138556 non-null  int64         
 9   NoOfMonths_PartBCov              138556 non-null  int64         
 10